In [6]:
import pandas as pd
import duckdb
import rank_bm25
import nltk

In [3]:
con = duckdb.connect()

df = con.execute("""
    SELECT *
    FROM read_parquet('../data/merged/merged_reviews_meta.parquet')
""").df()

df.head()

,rating,title,text,verified_purchase,product_title,average_rating,price,description,store,details
0,4.0,Keeps leather from drying out,I have a leather couch and because we live in ...,True,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...",4.8,None,[],Howard Products,"{""Package Dimensions"": ""7.1 x 5.5 x 3 inches; ..."
1,5.0,Five Stars,Grandson has had very good results with this p...,True,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,4.5,None,[],Yes To,"{""Item Form"": ""Powder"", ""Skin Type"": ""Acne Pro..."
2,5.0,Good Eye Patch,It is good. No problems. Eye patch is as adver...,True,Eye Patch Black Adult with Tie Band (6 Per Pack),4.4,None,[],Levine Health Products,"{""Manufacturer"": ""Levine Health Products""}"
3,5.0,Wonderful and super realistic,I have trichotillomania which is a hair pullin...,True,"Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4...",3.1,None,[],Cherioll,"{""Brand"": ""Cherioll"", ""Item Form"": ""Powder"", ""..."
4,5.0,Missing Review.......... It’s lost at Amazon S...,Where did my review go? I used your take a pic...,True,Precision Plunger Bars for Cartridge Grips – 9...,4.3,None,[The Precision Plunger Bars are designed to wo...,Precision,"{""UPC"": ""644287689178""}"


In [5]:
df["text"] = df["text"].fillna("").astype(str)
df.head()

,rating,title,text,verified_purchase,product_title,average_rating,price,description,store,details
0,4.0,Keeps leather from drying out,I have a leather couch and because we live in ...,True,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...",4.8,None,[],Howard Products,"{""Package Dimensions"": ""7.1 x 5.5 x 3 inches; ..."
1,5.0,Five Stars,Grandson has had very good results with this p...,True,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,4.5,None,[],Yes To,"{""Item Form"": ""Powder"", ""Skin Type"": ""Acne Pro..."
2,5.0,Good Eye Patch,It is good. No problems. Eye patch is as adver...,True,Eye Patch Black Adult with Tie Band (6 Per Pack),4.4,None,[],Levine Health Products,"{""Manufacturer"": ""Levine Health Products""}"
3,5.0,Wonderful and super realistic,I have trichotillomania which is a hair pullin...,True,"Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4...",3.1,None,[],Cherioll,"{""Brand"": ""Cherioll"", ""Item Form"": ""Powder"", ""..."
4,5.0,Missing Review.......... It’s lost at Amazon S...,Where did my review go? I used your take a pic...,True,Precision Plunger Bars for Cartridge Grips – 9...,4.3,None,[The Precision Plunger Bars are designed to wo...,Precision,"{""UPC"": ""644287689178""}"


In [7]:
nltk.download("stopwords")

from nltk.corpus import stopwords
stop_words = set(stopwords.words("english"))

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/arafat/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Writing the preprocessing function to tokenize the text

In [17]:
import re
def preprocess_text(text):

    # Lowercase
    text = text.lower()

    # Remove punctuation
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Remove whitespace
    tokens = text.split()

    # remove stop words
    tokens = [word for word in tokens if word not in stop_words]

    return tokens

In [18]:
df["tokens"] = df["text"].apply(preprocess_text)

In [19]:
df.head()

,rating,title,text,verified_purchase,product_title,average_rating,price,description,store,details,tokens
0,4.0,Keeps leather from drying out,I have a leather couch and because we live in ...,True,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...",4.8,None,[],Howard Products,"{""Package Dimensions"": ""7.1 x 5.5 x 3 inches; ...","[leather, couch, live, area, air, conditioning..."
1,5.0,Five Stars,Grandson has had very good results with this p...,True,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,4.5,None,[],Yes To,"{""Item Form"": ""Powder"", ""Skin Type"": ""Acne Pro...","[grandson, good, results, product, acne]"
2,5.0,Good Eye Patch,It is good. No problems. Eye patch is as adver...,True,Eye Patch Black Adult with Tie Band (6 Per Pack),4.4,None,[],Levine Health Products,"{""Manufacturer"": ""Levine Health Products""}","[good, problems, eye, patch, advertised, would..."
3,5.0,Wonderful and super realistic,I have trichotillomania which is a hair pullin...,True,"Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4...",3.1,None,[],Cherioll,"{""Brand"": ""Cherioll"", ""Item Form"": ""Powder"", ""...","[trichotillomania, hair, pulling, disorder, ey..."
4,5.0,Missing Review.......... It’s lost at Amazon S...,Where did my review go? I used your take a pic...,True,Precision Plunger Bars for Cartridge Grips – 9...,4.3,None,[The Precision Plunger Bars are designed to wo...,Precision,"{""UPC"": ""644287689178""}","[review, go, used, take, picture, link, came, ..."


## Build BM25

In [11]:
from rank_bm25 import BM25Okapi

In [20]:
tokenized_corpus = df["tokens"].tolist()
bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
import os
data_dir = "../data/processed"
os.makedirs(data_dir, exist_ok=True)
df.to_parquet(os.path.join(data_dir, "tokenized_docs.parquet"), index=False)


In [24]:
import pickle
os.makedirs("../models", exist_ok=True)
with open(os.path.join("../models", "bm25_model.pkl"), "wb") as f:
    pickle.dump(bm25, f)

with open(os.path.join("../data/processed", "tokenized_corpus.pkl"), "wb") as f:
    pickle.dump(tokenized_corpus, f)

## Search function

In [22]:
def search(query, top_k=10):
    tokenized_query = preprocess_text(query)
    scores = bm25.get_scores(tokenized_query)
    
    docs_copy = df.copy()
    docs_copy["score"] = scores
    
    return docs_copy.sort_values("score", ascending=False).head(top_k)

In [23]:
search("good battery life")

,rating,title,text,verified_purchase,product_title,average_rating,price,description,store,details,tokens,score
684744,5.0,Great shaver,"great shaver, good battery life, battery life...",True,Wahl Smart Shave Rechargeable lithium ion wet ...,4.2,None,[],WAHL,"{""Brand"": ""WAHL"", ""Recommended Uses For Produc...","[great, shaver, good, battery, life, battery, ...",19.842759
127376,5.0,It's ok 👌🙂 not to bad,Not a problem with it battery life is good,True,"Head Shaver, 5-in-1 Shaver for Bald Men with L...",4.2,None,[],Syllable,"{""Brand"": ""Syllable"", ""Recommended Uses For Pr...","[problem, battery, life, good]",18.737010
267740,5.0,Great Shaver all round cuts to the edge,Battery life good very easy to clean,True,"Head Shaver, 5-in-1 Shaver for Bald Men with L...",4.3,None,[],Syllable,"{""Brand"": ""Syllable"", ""Recommended Uses For Pr...","[battery, life, good, easy, clean]",17.993469
96538,5.0,Great Head Shaver,Good battery life and easy to use.,True,"Head Shaver, 5-in-1 Shaver for Bald Men with L...",4.2,None,[],Syllable,"{""Brand"": ""Syllable"", ""Recommended Uses For Pr...","[good, battery, life, easy, use]",17.993469
637253,5.0,Easy to use - trims well - no skin or hair pulls!,Need a quick close trim! This is for you! Qual...,True,Remington Virtually Indestructible All-in-One ...,4.4,None,[Engineered tough to last. The Remington virtu...,Remington,"{""Recommended Uses For Product"": ""Grooming"", ""...","[need, quick, close, trim, quality, battery, g...",17.781562
252616,5.0,Use it everyday,Good for shaving head. Good battery life. Batt...,True,Electric Shaver for Men Wet and Dry Waterproof...,4.1,None,[],Aesfee,"{""Brand"": ""Aesfee"", ""Power Source"": ""Battery P...","[good, shaving, head, good, battery, life, bat...",17.662118
560904,5.0,Good product,Good product- good battery life - easy to clean,True,Electric Shaver for Men Electric Razors for Ba...,4.1,None,[],AHGDB,"{""Brand"": ""AHGDB"", ""Power Source"": ""Battery Po...","[good, product, good, battery, life, easy, clean]",17.464784
306101,5.0,Great product!,The battery life is amazing!,True,Hatteker Cordless Hair Trimmer Pro Hair clippe...,4.2,None,[],Hatteker,"{""Is Discontinued By Manufacturer"": ""No"", ""Pac...","[battery, life, amazing]",16.425755
145306,5.0,It works as advertised,Easy to use and has a good battery life still ...,True,Blackhead Remover Vacuum Blackhead removal too...,3.8,None,[],Alecoy,"{""Brand"": ""Alecoy"", ""Item Form"": ""Pen"", ""Mater...","[easy, use, good, battery, life, still, works,...",16.079251
144994,5.0,Did the job,So far the battery life has been good. It has ...,False,Blackhead Remover Vacuum Blackhead removal too...,3.8,None,[],Alecoy,"{""Brand"": ""Alecoy"", ""Item Form"": ""Pen"", ""Mater...","[far, battery, life, good, almost, 1, yr, half]",16.079251
